In [2]:
import numpy as np, pandas as pd, time
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

d = np.load("../data/processed/fd001_windows.npz")
X_tr, y_tr, X_val, y_val = d["X_tr"], d["y_tr"], d["X_val"], d["y_val"]
X_test, y_test = d["X_test"], d["y_test"]
features = list(d["features"])
y_test = np.minimum(y_test, 125)   # same cap as training, for a fair comparison

def rmse(y, p):
    return float(np.sqrt(np.mean((y - p) ** 2)))

def nasa_score(y, p):
    e = p - y   # late (e>0) is penalised more than early
    return float(np.sum(np.where(e < 0, np.exp(-e / 13) - 1, np.exp(e / 10) - 1)))

results = {}

In [3]:
def summarize(X):
    t = np.arange(X.shape[1])
    slope = ((X - X.mean(1, keepdims=True)) * (t - t.mean())[None, :, None]).sum(1) / ((t - t.mean()) ** 2).sum()
    return np.concatenate([X[:, -1], X.mean(1), X.std(1), X.min(1), X.max(1), slope], axis=1)

F_tr, F_val, F_test = summarize(X_tr), summarize(X_val), summarize(X_test)
print(F_tr.shape)

(14241, 90)


In [4]:
t0 = time.time()
rf = RandomForestRegressor(n_estimators=300, min_samples_leaf=3, n_jobs=-1, random_state=42)
rf.fit(F_tr, y_tr)
results["RandomForest"] = {
    "val_RMSE": rmse(y_val, rf.predict(F_val)),
    "test_RMSE": rmse(y_test, rf.predict(F_test)),
    "test_NASA": nasa_score(y_test, rf.predict(F_test)),
    "time_s": time.time() - t0,
}
print(results["RandomForest"])

{'val_RMSE': 12.876663208447077, 'test_RMSE': 12.910768910034514, 'test_NASA': 278.26595067876985, 'time_s': 59.17059636116028}


In [5]:
t0 = time.time()
xgb = XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6,
                   subsample=0.8, colsample_bytree=0.8, n_jobs=-1, random_state=42)
xgb.fit(F_tr, y_tr)
results["XGBoost"] = {
    "val_RMSE": rmse(y_val, xgb.predict(F_val)),
    "test_RMSE": rmse(y_test, xgb.predict(F_test)),
    "test_NASA": nasa_score(y_test, xgb.predict(F_test)),
    "time_s": time.time() - t0,
}
print(results["XGBoost"])
pd.DataFrame(results).T

{'val_RMSE': 12.55864143371582, 'test_RMSE': 12.565442085266113, 'test_NASA': 260.9076843261719, 'time_s': 7.95393443107605}


,val_RMSE,test_RMSE,test_NASA,time_s
RandomForest,12.876663,12.910769,278.265951,59.170596
XGBoost,12.558641,12.565442,260.907684,7.953934


In [6]:
y_test_raw = d["y_test"]   # uncapped true RUL
for name, m in [("RandomForest", rf), ("XGBoost", xgb)]:
    p = m.predict(F_test)
    print(name, "uncapped test RMSE:", round(rmse(y_test_raw, p), 2), "| NASA:", round(nasa_score(y_test_raw, p), 1))

pd.DataFrame(results).T.to_csv("../results/baselines.csv")

RandomForest uncapped test RMSE: 14.11 | NASA: 311.0
XGBoost uncapped test RMSE: 13.72 | NASA: 288.6


In [7]:
for name, m in [("RandomForest", rf), ("XGBoost", xgb)]:
    p = m.predict(F_test)
    results[name]["test_RMSE_raw"] = rmse(y_test_raw, p)
    results[name]["test_NASA_raw"] = nasa_score(y_test_raw, p)
pd.DataFrame(results).T.to_csv("../results/baselines.csv")
pd.DataFrame(results).T.round(2)

,val_RMSE,test_RMSE,test_NASA,time_s,test_RMSE_raw,test_NASA_raw
RandomForest,12.88,12.91,278.27,59.17,14.11,310.99
XGBoost,12.56,12.57,260.91,7.95,13.72,288.56
